# CBOW gradients, checked by finite differences

CBOW predicts the word at the center of a window from the average of the vectors of the
words around it. You will write the loss of one window, a finite-difference gradient that
works for any function of an array, and the exact gradient of the loss with respect to
both embedding tables. Then the finite differences test the exact gradient, the check
that would have caught the mistake in the standard CBOW code.

**How to work through it.** Each part asks for one function. Replace the `TODO` and the
`raise NotImplementedError` with your code, then run the check cell under it. The check
prints `Part N passed` when your function is right, and stops with a message that says
what is wrong when it is not.

## Setup

Run this cell first.

In [ ]:
import numpy as np

# Eight words in three dimensions. Row i of V_in is word i's vector as a context word,
# row i of U_out its vector as the word to predict. Both start small and random.
vocab = ["the", "kettle", "boils", "water", "for", "tea", "cold", "pours"]
idx = {w: i for i, w in enumerate(vocab)}
rng = np.random.default_rng(5)
V_in = rng.normal(0.0, 0.5, size=(len(vocab), 3))
U_out = rng.normal(0.0, 0.5, size=(len(vocab), 3))

# One window: the center word "water" with the three words on each side of it.
sentence = "the kettle boils water for the tea".split()
center = idx["water"]
context = [idx[w] for w in sentence[:3] + sentence[4:]]
print("context ids", context, " center id", center)

## Part 1 (coding task): The loss of one window

Write `cbow_loss(V_in, U_out, context, center)`. `context` is the list of the window's word
ids, one per position, and `center` is the id of the center word. Average the rows of
`V_in` at the context positions into $\mathbf{h}$, score every word $w$ of the vocabulary
by $\mathbf{u}_w^\top \mathbf{h}$, where $\mathbf{u}_w$ is row $w$ of `U_out`, and return
$-\ln P(\text{center} \mid \text{context})$ under the softmax of those scores, as a float.

In [ ]:
def cbow_loss(V_in, U_out, context, center):
    """-ln P(center | context) for CBOW with a full softmax."""
    # TODO
    raise NotImplementedError

In [ ]:
loss = cbow_loss(V_in, U_out, context, center)
assert np.ndim(loss) == 0, "return a single number"
assert abs(loss - 2.135094006347772) < 1e-9, f"loss {float(loss):.6f} on the setup window, expected 2.135094"
assert abs(cbow_loss(V_in, np.zeros_like(U_out), context, center) - np.log(8)) < 1e-12, "with every score 0 the softmax is uniform over the 8 words, so the loss should be ln 8"
assert abs(cbow_loss(V_in, U_out, [idx["kettle"], idx["boils"]], idx["tea"]) - 1.8848277938066138) < 1e-9, "wrong loss for the window kettle, boils with center tea"
big = cbow_loss(60 * V_in, 60 * U_out, context, center)
assert np.isfinite(big), "the loss is not finite with the tables scaled by 60, where the largest score is about 1,400: exp(1400) overflows"
assert abs(big - 1033.0995226694267) < 1e-6, f"loss {float(big):.4f} with the tables scaled by 60, expected 1033.0995"
print("Part 1 passed")

## Part 2 (coding task): A finite-difference gradient

Write `numerical_gradient(f, A, eps=1e-5)` for a two-dimensional array `A` and a function
`f` that takes an array shaped like `A` and returns a number. Return an array `G` shaped
like `A` whose entries estimate the derivatives of `f` by the central difference

$$
G_{ij} = \frac{f(A + \varepsilon E_{ij}) - f(A - \varepsilon E_{ij})}{2\varepsilon},
$$

where $E_{ij}$ is 1 at position $(i, j)$ and 0 everywhere else.

In [ ]:
def numerical_gradient(f, A, eps=1e-5):
    """Gradient of f at A from central differences, nudging one entry at a time."""
    # TODO
    raise NotImplementedError

In [ ]:
A = np.random.default_rng(0).normal(size=(3, 4))
A_before = A.copy()
G = numerical_gradient(lambda M: float(np.sum(M ** 3)), A, eps=1e-5)
assert np.abs(A - A_before).max() < 1e-12, "numerical_gradient changed A: put each entry back after nudging it"
assert np.shape(G) == A.shape, f"G has shape {np.shape(G)}, expected {A.shape}"
exact = 3 * A ** 2
assert not np.allclose(G, 2 * exact, rtol=1e-4), "G is twice the gradient: the two points are 2 eps apart"
err = np.abs(G - exact).max()
assert err < 1e-7, f"largest error {err:.1e} against the exact gradient 3A^2 of sum(A^3): a central difference is off by about 1e-10 here, a one-sided one by about 1e-5"
P = np.array([[2.0, -1.0], [0.5, 3.0]])
G = numerical_gradient(lambda M: float(M[0, 0] * M[1, 1] - M[0, 1] * M[1, 0]), P)
assert np.allclose(G, [[3.0, -0.5], [1.0, 2.0]], atol=1e-8), "wrong gradient for the determinant of a 2 x 2 matrix, or the default eps is missing"
print("Part 2 passed")

## Part 3 (coding task): The exact gradient

Write `cbow_gradients(V_in, U_out, context, center)`, which returns the pair
`(dV_in, dU_out)`: the gradient of `cbow_loss` with respect to every entry of `V_in` and
with respect to every entry of `U_out`, as arrays shaped like `V_in` and `U_out`. Derive it
by hand, then write it with NumPy, without calling `numerical_gradient`. The check compares
your arrays with `numerical_gradient` on the window from the setup and on two more windows.

In [ ]:
def cbow_gradients(V_in, U_out, context, center):
    """Gradients of cbow_loss with respect to V_in and U_out."""
    # TODO
    raise NotImplementedError

In [ ]:
windows = [
    (context, center),
    ([idx["kettle"], idx["boils"]], idx["tea"]),
    ([idx["pours"], idx["the"], idx["cold"], idx["the"]], idx["water"]),
]
for ctx, c in windows:
    name = " ".join(vocab[i] for i in ctx) + " -> " + vocab[c]
    dV, dU = cbow_gradients(V_in, U_out, ctx, c)
    assert np.shape(dV) == V_in.shape and np.shape(dU) == U_out.shape, "return two arrays shaped like V_in and U_out"
    nV = numerical_gradient(lambda M: cbow_loss(M, U_out, ctx, c), V_in.copy())
    nU = numerical_gradient(lambda M: cbow_loss(V_in, M, ctx, c), U_out.copy())
    assert np.allclose(dU, nU, atol=1e-7), f"dU_out does not match the finite differences for the window {name}"
    outside = [i for i in range(len(vocab)) if i not in ctx]
    assert np.allclose(dV[outside], 0.0), f"dV_in has nonzero rows for words outside the window {name}"
    k = len(ctx)
    once = [i for i in set(ctx) if ctx.count(i) == 1]
    assert not np.allclose(dV[once], k * nV[once], atol=1e-7), f"the context rows of dV_in are {k} times the finite differences for the window {name}: h is the average of {k} vectors"
    assert np.allclose(dV[once], nV[once], atol=1e-7), f"dV_in does not match the finite differences for the window {name}"
    for i in set(ctx) - set(once):
        assert np.allclose(dV[i], nV[i], atol=1e-7), f"the row of '{vocab[i]}', which fills {ctx.count(i)} positions of the window {name}, does not match the finite differences"
print("Part 3 passed")

## Solution

### Part 1: The loss of one window

In [ ]:
def cbow_loss(V_in, U_out, context, center):
    """-ln P(center | context) for CBOW with a full softmax."""
    h = V_in[context].mean(axis=0)
    scores = U_out @ h
    top = scores.max()
    return float(top + np.log(np.exp(scores - top).sum()) - scores[center])

**Why it works.** The softmax does not change when the same number is subtracted from every score, so
subtracting the largest score $m$ leaves exponents of at most 0 and nothing overflows. The
loss is then $m + \ln \sum_w e^{s_w - m} - s_{\text{center}}$, the log-sum-exp form. With
the tables scaled by 60 the largest score is about 1,400, and `np.exp(1400)` is `inf` in
float64, so the unshifted formula returns `inf`. Indexing `V_in[context]` takes one row per
position, so a word that fills two positions counts twice in the average, as it should.

### Part 2: A finite-difference gradient

In [ ]:
def numerical_gradient(f, A, eps=1e-5):
    """Gradient of f at A from central differences, nudging one entry at a time."""
    G = np.zeros(A.shape)
    for i in range(A.shape[0]):
        for j in range(A.shape[1]):
            kept = A[i, j]
            A[i, j] = kept + eps
            up = f(A)
            A[i, j] = kept - eps
            down = f(A)
            A[i, j] = kept
            G[i, j] = (up - down) / (2 * eps)
    return G

**Why it works.** Each entry is nudged up and down while every other entry stays put, which is the partial
derivative's definition with a finite step. A Taylor expansion of $f$ around $A$ shows why
the central difference is the one to use: the second-derivative terms of $f(A + \varepsilon E)$
and $f(A - \varepsilon E)$ cancel, so the error is of order $\varepsilon^2$, about
$10^{-10}$ here, while a one-sided difference keeps an error of about
$\tfrac{\varepsilon}{2} f''$, about $10^{-5}$. Putting the entry back matters because `f`
reads the whole array, and an entry left nudged would shift every later estimate. The
method costs two calls of `f` per entry, so it is a test for a derived gradient, not a way
to train.

### Part 3: The exact gradient

In [ ]:
def cbow_gradients(V_in, U_out, context, center):
    """Gradients of cbow_loss with respect to V_in and U_out."""
    h = V_in[context].mean(axis=0)
    scores = U_out @ h
    p = np.exp(scores - scores.max())
    p /= p.sum()
    p[center] -= 1.0                       # dLoss/dscores = softmax - one-hot
    dU_out = np.outer(p, h)
    dh = U_out.T @ p
    dV_in = np.zeros_like(V_in)
    np.add.at(dV_in, context, dh / len(context))   # one share per position, repeats add up
    return dV_in, dU_out

**Why it works.** With $\mathbf{p}$ the softmax of the scores and $\mathbf{y}$ the one-hot vector of the
center word, the loss has gradient $\mathbf{p} - \mathbf{y}$ with respect to the scores
$\mathbf{s} = U_{\text{out}} \mathbf{h}$. Score $w$ is $\mathbf{u}_w^\top \mathbf{h}$, so
row $w$ of `dU_out` is $(p_w - y_w)\,\mathbf{h}$, the outer product, and
$\partial \mathcal{L} / \partial \mathbf{h} = U_{\text{out}}^\top (\mathbf{p} - \mathbf{y})$.
$\mathbf{h}$ is the mean of the $k$ context rows, so each position passes $1/k$ of that
gradient to its word's row, and a word in two positions, like "the" in the setup window,
collects $2/k$. `dV_in[context] += dh / k` would get that wrong: with a repeated id NumPy
writes the row once, so the second share is lost. `np.add.at` adds every share.

word2vec.c, Gensim and fastText add the full $\partial \mathcal{L} / \partial \mathbf{h}$ to
every context row, without the $1/k$. On the setup window that is 6 times the gradient
above, and the assertion against $k$ times the finite differences catches exactly that
version.